In [2]:
import numpy as np
from typing import List, Optional
from dataclasses import dataclass


@dataclass
class RankingResult:
    """Data class to hold ranking results"""
    item_id: str
    relevance_score: float
    position: int


class MetricsCalculator:
    """
    Class to calculate various ranking metrics:
    - CG (Cumulative Gain)
    - DCG (Discounted Cumulative Gain)
    - NDCG (Normalized Discounted Cumulative Gain)
    - PNR (Position Normalized Rank)
    """
    
    @staticmethod
    def cg(relevance_scores: List[float], k: Optional[int] = None) -> float:
        """
        Calculate Cumulative Gain (CG)
        
        CG = sum of relevance scores up to position k
        
        Args:
            relevance_scores: List of relevance scores
            k: Number of top results to consider (None = all)
            
        Returns:
            Cumulative Gain value
        """
        if k is None:
            k = len(relevance_scores)
        return sum(relevance_scores[:k])
    
    @staticmethod
    def dcg(relevance_scores: List[float], k: Optional[int] = None) -> float:
        """
        Calculate Discounted Cumulative Gain (DCG)
        
        DCG = sum(relevance[i] / log2(i + 1)) for i in range(k)
        
        Args:
            relevance_scores: List of relevance scores
            k: Number of top results to consider (None = all)
            
        Returns:
            Discounted Cumulative Gain value
        """
        if k is None:
            k = len(relevance_scores)
        
        dcg_score = 0.0
        for i in range(min(k, len(relevance_scores))):
            if i == 0:
                dcg_score += relevance_scores[i]
            else:
                dcg_score += relevance_scores[i] / np.log2(i + 1)
        return dcg_score
    
    @staticmethod
    def ideal_dcg(relevance_scores: List[float], k: Optional[int] = None) -> float:
        """
        Calculate Ideal DCG (IDCG) - DCG of perfectly sorted results
        
        Args:
            relevance_scores: List of relevance scores
            k: Number of top results to consider (None = all)
            
        Returns:
            Ideal Discounted Cumulative Gain value
        """
        if k is None:
            k = len(relevance_scores)
        
        # Sort in descending order to get ideal ranking
        sorted_scores = sorted(relevance_scores, reverse=True)
        return MetricsCalculator.dcg(sorted_scores, k)
    
    @staticmethod
    def ndcg(relevance_scores: List[float], k: Optional[int] = None) -> float:
        """
        Calculate Normalized Discounted Cumulative Gain (NDCG)
        
        NDCG = DCG / IDCG
        
        Args:
            relevance_scores: List of relevance scores
            k: Number of top results to consider (None = all)
            
        Returns:
            Normalized Discounted Cumulative Gain value (0-1)
        """
        dcg_score = MetricsCalculator.dcg(relevance_scores, k)
        idcg_score = MetricsCalculator.ideal_dcg(relevance_scores, k)
        
        if idcg_score == 0:
            return 0.0
        
        return dcg_score / idcg_score
    
    @staticmethod
    def pnr(relevance_scores: List[float], k: Optional[int] = None) -> float:
        """
        Calculate Position Normalized Rank (PNR)
        
        PNR = 1 - (average_position_of_relevant_items - 1) / (n - 1)
        where relevant items are those with score > 0
        
        Args:
            relevance_scores: List of relevance scores
            k: Number of top results to consider (None = all)
            
        Returns:
            Position Normalized Rank value (0-1, higher is better)
        """
        if k is None:
            k = len(relevance_scores)
        
        # Find positions of relevant items (score > 0)
        relevant_positions = [i + 1 for i, score in enumerate(relevance_scores[:k]) if score > 0]
        
        if not relevant_positions:
            return 0.0
        
        avg_position = np.mean(relevant_positions)
        n = k
        
        if n == 1:
            return 1.0
        
        pnr_score = 1 - (avg_position - 1) / (n - 1)
        return max(0.0, pnr_score)  # Ensure non-negative
    
    def calculate_all_metrics(self, relevance_scores: List[float], k: Optional[int] = None) -> dict:
        """
        Calculate all metrics for given relevance scores
        
        Args:
            relevance_scores: List of relevance scores
            k: Number of top results to consider (None = all)
            
        Returns:
            Dictionary with all metric values
        """
        return {
            'CG': self.cg(relevance_scores, k),
            'DCG': self.dcg(relevance_scores, k),
            'IDCG': self.ideal_dcg(relevance_scores, k),
            'NDCG': self.ndcg(relevance_scores, k),
            'PNR': self.pnr(relevance_scores, k)
        }


In [3]:
class ToyDataGenerator:
    """
    Class to generate toy data for ranking metrics calculation.
    Generates various scenarios to test ranking algorithms.
    """
    
    @staticmethod
    def generate_perfect_ranking(n: int = 10, max_relevance: float = 5.0) -> List[float]:
        """
        Generate a perfect ranking (scores in descending order)
        
        Args:
            n: Number of items
            max_relevance: Maximum relevance score
            
        Returns:
            List of relevance scores in perfect order
        """
        scores = np.linspace(max_relevance, 0, n)
        return scores.tolist()
    
    @staticmethod
    def generate_random_ranking(n: int = 10, max_relevance: float = 5.0, 
                               seed: Optional[int] = None) -> List[float]:
        """
        Generate a random ranking
        
        Args:
            n: Number of items
            max_relevance: Maximum relevance score
            seed: Random seed for reproducibility
            
        Returns:
            List of relevance scores in random order
        """
        if seed is not None:
            np.random.seed(seed)
        scores = np.random.uniform(0, max_relevance, n)
        return scores.tolist()
    
    @staticmethod
    def generate_worst_ranking(n: int = 10, max_relevance: float = 5.0) -> List[float]:
        """
        Generate worst ranking (scores in ascending order - worst first)
        
        Args:
            n: Number of items
            max_relevance: Maximum relevance score
            
        Returns:
            List of relevance scores in worst order
        """
        scores = np.linspace(0, max_relevance, n)
        return scores.tolist()
    
    @staticmethod
    def generate_binary_relevance(n: int = 10, num_relevant: int = 5,
                                  relevant_score: float = 1.0,
                                  seed: Optional[int] = None) -> List[float]:
        """
        Generate binary relevance scores (0 or 1)
        
        Args:
            n: Number of items
            num_relevant: Number of relevant items
            relevant_score: Score for relevant items (default 1.0)
            seed: Random seed for reproducibility
            
        Returns:
            List of binary relevance scores
        """
        if seed is not None:
            np.random.seed(seed)
        
        scores = [0.0] * n
        relevant_indices = np.random.choice(n, size=min(num_relevant, n), replace=False)
        for idx in relevant_indices:
            scores[idx] = relevant_score
        
        return scores
    
    @staticmethod
    def generate_graded_relevance(n: int = 10, grades: List[int] = [0, 1, 2, 3, 4, 5],
                                 seed: Optional[int] = None) -> List[float]:
        """
        Generate graded relevance scores (multiple relevance levels)
        
        Args:
            n: Number of items
            grades: List of possible relevance grades
            seed: Random seed for reproducibility
            
        Returns:
            List of graded relevance scores
        """
        if seed is not None:
            np.random.seed(seed)
        
        scores = np.random.choice(grades, size=n)
        return scores.tolist()
    
    @staticmethod
    def generate_realistic_search_results(n: int = 10, 
                                         high_relevance_prob: float = 0.2,
                                         max_relevance: float = 5.0,
                                         seed: Optional[int] = None) -> List[float]:
        """
        Generate realistic search results with some high-relevance items
        
        Args:
            n: Number of items
            high_relevance_prob: Probability of high relevance item
            max_relevance: Maximum relevance score
            seed: Random seed for reproducibility
            
        Returns:
            List of realistic relevance scores
        """
        if seed is not None:
            np.random.seed(seed)
        
        scores = []
        for _ in range(n):
            if np.random.random() < high_relevance_prob:
                # High relevance item
                scores.append(np.random.uniform(max_relevance * 0.7, max_relevance))
            else:
                # Low to medium relevance item
                scores.append(np.random.uniform(0, max_relevance * 0.5))
        
        return scores
    
    @staticmethod
    def generate_with_position_bias(n: int = 10, max_relevance: float = 5.0,
                                   bias_strength: float = 0.3,
                                   seed: Optional[int] = None) -> List[float]:
        """
        Generate scores with position bias (earlier positions tend to be more relevant)
        
        Args:
            n: Number of items
            max_relevance: Maximum relevance score
            bias_strength: Strength of position bias (0-1)
            seed: Random seed for reproducibility
            
        Returns:
            List of relevance scores with position bias
        """
        if seed is not None:
            np.random.seed(seed)
        
        scores = []
        for i in range(n):
            # Base score decreases with position
            base_score = max_relevance * (1 - bias_strength * i / n)
            # Add some randomness
            noise = np.random.uniform(-max_relevance * 0.2, max_relevance * 0.2)
            score = max(0, base_score + noise)
            scores.append(score)
        
        return scores
    
    def generate_all_scenarios(self, n: int = 10) -> dict:
        """
        Generate all different ranking scenarios
        
        Args:
            n: Number of items
            
        Returns:
            Dictionary with different ranking scenarios
        """
        return {
            'perfect': self.generate_perfect_ranking(n),
            'worst': self.generate_worst_ranking(n),
            'random': self.generate_random_ranking(n, seed=42),
            'binary': self.generate_binary_relevance(n, num_relevant=5, seed=42),
            'graded': self.generate_graded_relevance(n, seed=42),
            'realistic': self.generate_realistic_search_results(n, seed=42),
            'position_bias': self.generate_with_position_bias(n, seed=42)
        }


In [4]:
# Example usage: Generate toy data and calculate metrics

# Initialize classes
metrics_calc = MetricsCalculator()
data_gen = ToyDataGenerator()

# Generate different ranking scenarios
scenarios = data_gen.generate_all_scenarios(n=10)

# Calculate metrics for each scenario
print("=" * 80)
print("RANKING METRICS COMPARISON")
print("=" * 80)

for scenario_name, relevance_scores in scenarios.items():
    print(f"\n{scenario_name.upper()} Ranking:")
    print(f"  Relevance Scores: {[f'{s:.2f}' for s in relevance_scores]}")
    
    metrics = metrics_calc.calculate_all_metrics(relevance_scores)
    print(f"  CG:   {metrics['CG']:.4f}")
    print(f"  DCG:  {metrics['DCG']:.4f}")
    print(f"  IDCG: {metrics['IDCG']:.4f}")
    print(f"  NDCG: {metrics['NDCG']:.4f}")
    print(f"  PNR:  {metrics['PNR']:.4f}")

print("\n" + "=" * 80)


RANKING METRICS COMPARISON

PERFECT Ranking:
  Relevance Scores: ['5.00', '4.44', '3.89', '3.33', '2.78', '2.22', '1.67', '1.11', '0.56', '0.00']
  CG:   25.0000
  DCG:  16.7600
  IDCG: 16.7600
  NDCG: 1.0000
  PNR:  0.5556

WORST Ranking:
  Relevance Scores: ['0.00', '0.56', '1.11', '1.67', '2.22', '2.78', '3.33', '3.89', '4.44', '5.00']
  CG:   25.0000
  DCG:  9.5124
  IDCG: 16.7600
  NDCG: 0.5676
  PNR:  0.4444

RANDOM Ranking:
  Relevance Scores: ['1.87', '4.75', '3.66', '2.99', '0.78', '0.78', '0.29', '4.33', '3.01', '3.54']
  CG:   26.0068
  DCG:  14.6308
  IDCG: 16.8768
  NDCG: 0.8669
  PNR:  0.5000

BINARY Ranking:
  Relevance Scores: ['1.00', '1.00', '0.00', '0.00', '0.00', '1.00', '0.00', '1.00', '1.00', '0.00']
  CG:   5.0000
  DCG:  3.0357
  IDCG: 3.5616
  NDCG: 0.8523
  PNR:  0.5333

GRADED Ranking:
  Relevance Scores: ['3.00', '4.00', '2.00', '4.00', '4.00', '1.00', '2.00', '2.00', '2.00', '4.00']
  CG:   28.0000
  DCG:  15.5855
  IDCG: 16.9005
  NDCG: 0.9222
  PNR:  0.50

In [5]:
# Example: Calculate metrics at different k values (top-k)

example_scores = data_gen.generate_realistic_search_results(n=15, seed=123)
print("Example Relevance Scores:", [f'{s:.2f}' for s in example_scores])
print("\nMetrics at different k values:")
print("-" * 80)

for k in [3, 5, 10, 15]:
    metrics = metrics_calc.calculate_all_metrics(example_scores, k=k)
    print(f"\nk={k}:")
    print(f"  CG:   {metrics['CG']:.4f}")
    print(f"  DCG:  {metrics['DCG']:.4f}")
    print(f"  NDCG: {metrics['NDCG']:.4f}")
    print(f"  PNR:  {metrics['PNR']:.4f}")

print("\n" + "=" * 80)


Example Relevance Scores: ['0.72', '1.38', '1.06', '1.71', '0.98', '1.82', '0.15', '1.84', '3.76', '1.33', '2.12', '1.53', '0.81', '0.57', '1.58']

Metrics at different k values:
--------------------------------------------------------------------------------

k=3:
  CG:   3.1514
  DCG:  2.7610
  NDCG: 0.3916
  PNR:  0.5000

k=5:
  CG:   5.8438
  DCG:  4.0392
  NDCG: 0.4643
  PNR:  0.5000

k=10:
  CG:   14.7533
  DCG:  6.9999
  NDCG: 0.6334
  PNR:  0.5000

k=15:
  CG:   21.3600
  DCG:  8.8116
  NDCG: 0.7379
  PNR:  0.5000

